# Notebook 2: Functional Neural Modules

JAX is purely functional, meaning functions cannot modify global state or class attributes. Thus, JAX neural network libraries provide wrappers that construct 'stateless' forward passes. 
Simply provides a lightweight alternative to libraries like Flax/Haiku via its `module.py` abstractions.

Run the cell below to configure the `simply` module path.

In [13]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), '.', 'third-party', 'simply'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import jax
import jax.numpy as jnp
from simply.utils import module
module.ModuleRegistry.OVERWRITE_DUPLICATE = True
from simply.utils import initializer
print("Ready to use Simply modules!")

Ready to use Simply modules!


## 1. Einsum Intuition
Before we dive into Modules, we need to understand `jnp.einsum`. The Simply codebase relies heavily on Einstein summation for concise operations across highly-sharded multi-dimensional tensors. It's heavily used in `EinsumLinear`.

### Exercise 1: Build an Einsum Matrix Multiplication.
A standard batch multiplication of an input `[B, S, D]` mapped to `[D, H]` takes the form `jnp.dot` or `@`. Create an equivalent operation strictly via `jnp.einsum`.

In [7]:
# Exercise 1 Code

# B: batch, S: seq, D: model_dim, H: hidden_dim
inputs = jnp.ones((2, 4, 16)) # [B, S, D]
weights = jax.random.normal(jax.random.PRNGKey(0), (16, 32)) # [D, H]

# TODO: Replace the standard matrix multiplication below with `jnp.einsum`.
standard_out = inputs @ weights

einsum_out = jnp.einsum('bsd,dh->bsh', inputs, weights)

if einsum_out is not None:
    print(f"Einsum shape output: {einsum_out.shape} -> Expected (2, 4, 32)")
    assert(jnp.allclose(standard_out, einsum_out))

Einsum shape output: (2, 4, 32) -> Expected (2, 4, 32)


## 2. Stateless Module Initialization
Simply's modules inherit from `simply.utils.module.SimplyModule`. Crucially, they possess two independent functions: `init(prng_key)` and `apply(params, x)`. They *do not* store `params` inside the class tree during runtime (unlike PyTorch).

### Exercise 2: Implementing a Subclass and Init
Subclass `SimplyModule` to build `.init()`. Return a simple dictionary tree containing an array of shapes instantiated via `prng_key` using a Xavier Uniform Initializer from `simply.utils.initializer`.

In [14]:
# Exercise 2 Code

from collections import abc
import dataclasses

# Because SimplyModule is meant to be subclassed functionally via config dictionaries,
# variables are instantiated structurally using `dataclasses.dataclass`.
@module.ModuleRegistry.register
@dataclasses.dataclass
class CustomDense(module.SimplyModule):
    dim_in: int
    dim_out: int
    
    def init(self, prng_key):
        # TODO: create a parameter dictionary mapping 'weight': <random array [self.dim_in, self.dim_out]>
        # Use simply.utils.initializer.XavierUniformInit()
        init_fn = initializer.XavierUniformInit()
        weight = init_fn(prng_key, (self.dim_in, self.dim_out), dim_annotation='io', dtype=jnp.float32)
        return {'weight': weight} 
        
    def apply(self, params, x):
        return jnp.einsum('...i,io->...o', x, params)

In [16]:
prng_key = jax.random.PRNGKey(0)
# Test initialization
dense_layer = CustomDense(dim_in=16, dim_out=32)
layer_params = dense_layer.init(prng_key=prng_key)

if layer_params is not None:
    print("Layer Initialized Params:", layer_params.keys())

Layer Initialized Params: dict_keys(['weight'])


## 3. Stateless Formward Pass
Building off Exercise 2, you're going to use your `CustomDense` layer statelessly.

### Exercise 3: Applying Parameters
Complete the `.apply(params, x)` function of the layer above (or create a new dummy layer here). Ensure you unpack the parameter dictionary mapping back to the logic mapping you established via initialization.

In [18]:
# Exercise 3 Code

@module.ModuleRegistry.register
@dataclasses.dataclass
class CustomDenseComplete(CustomDense):
    def apply(self, params, x):
        # TODO: Implement a forward pass `x @ weight` extracting `weight` from the `params` dict
        return jnp.einsum('...i,io->...o', x, params['weight'])
        
dense_complete = CustomDenseComplete(dim_in=16, dim_out=32)
layer_params = dense_complete.init(jax.random.PRNGKey(1))

test_input = jnp.ones((4, 16))
# TODO: Execute a forward pass using the initialized tree!
forward_pass_output = dense_complete.apply(layer_params, test_input)

if forward_pass_output is not None:
    print(f"Output of functional pass shape: {forward_pass_output.shape}")

Output of functional pass shape: (4, 32)


## 4. `EinsumLinear`
In Simply, dense layers are usually wrapped around the `module.EinsumLinear` utility. It handles biases, mixed-precision settings (`activation_dtype`), weight initialization, and automatically applies `PartitionSpec` metadata to the `AnnotatedArray` parameters it issues via `init`.

### Exercise 4: Constructing EinsumLinear
Construct an `EinsumLinear` instance specifying an equation that takes `(..., input_dim)` and a `(input_dim, output_dim)` weight and returns `(..., output_dim)`. You'll need to define `eqn` strings correctly.

In [21]:
# Exercise 4 Code

# A hint for simply! EinsumLinear requires a 1D bias representation if used.
linear_layer = getattr(module, 'EinsumLinear', None)

# TODO: Setup `module.EinsumLinear` initialized below. 
# Ensure equation maps variable arbitrary leading dims (...) along with the final dim `i` to output dim `o`.
# `eqn` string format: e.g. <input>,<weight>-><output>
# Hint: "...i,io->...o" is the standard formula for Simply's FFN
my_linear = linear_layer(
   eqn='io,...i->...o',
   weight_shape=(16, 32),
)

if my_linear is not None:
   my_params = my_linear.init(jax.random.PRNGKey(42))
   out = my_linear.apply(my_params, jnp.ones((2, 10, 16)))
   print("Executed EinsumLinear:", out.shape)

Executed EinsumLinear: (2, 10, 32)


## 5. Wrapping a `LayerNorm` pipeline
Finally, we will glue multiple operations statelessly inside a parent module. This mimics the construction of Transformer layers in Simply.

### Exercise 5: Wrapping Multiple Modules
Using the `LayerNorm` and `EinsumLinear` abstractions from `simply.model_lib`, create an MLP parent class. Provide its `init` function recursively tracking parameters of its children module layers.

In [25]:
# Exercise 5 Code

from simply.model_lib import LayerNorm, EinsumLinear

@module.ModuleRegistry.register  # Simply makes layers registerable
@dataclasses.dataclass
class SimpleMLP(module.SimplyModule):
    # TODO 1: Initialize hyper-params as dataclass variables
    dim: int

    def setup(self):
        # Setup runs once per instantiate. Here you instantiate child subclasses.
        self.norm = LayerNorm(dim=self.dim)
        self.mlp = EinsumLinear(
            eqn='io,...i->...o',
            weight_shape=[self.dim, self.dim],
        )
        
    def init(self, prng_key):
        # TODO 2: Call the `init` on child instances generating independent PRNG Keys 
        # using jax.random.split and map their outputs to `norm_params`, `mlp_params`.
        k1, k2 = jax.random.split(prng_key, 2)
        norm_params = self.norm.init(k1)
        mlp_params = self.mlp.init(k2)
        return {
            'norm_layer': norm_params,
            'mlp_layer': mlp_params,
        }
        
    def apply(self, params, x):
        x = self.norm.apply(params['norm_layer'], x)
        x = self.mlp.apply(params['mlp_layer'], x)
        return x

mlp = SimpleMLP(dim=128)
my_mlp_params = mlp.init(jax.random.PRNGKey(4))

if my_mlp_params is not None:
    print("Top-level Param Keys:", my_mlp_params.keys())
    print("Nested MLP Keys:", my_mlp_params['mlp_layer'].keys())

 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0.]
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0.]


Top-level Param Keys: dict_keys(['norm_layer', 'mlp_layer'])
Nested MLP Keys: dict_keys(['w'])


---

## ✨ Bonus Material: Additional Topics & Exercises

The following exercises and concepts were merged from the previous `02_model_layers_and_einsum.ipynb` curriculum.

# 02: Model Layers & Einsum in Simply
Now that you understand pure functional modules, let's explore how `simply` constructs the atomic layers of LLMs, such as `LayerNorm` and linear layers.

To build large language models effectively, `simply` relies heavily on `einsum` (Einstein Summation) notation instead of rigid operations like `np.matmul` or PyTorch's `nn.Linear`.

In [26]:
import sys
import os
sys.path.append(os.path.abspath('./third-party/simply'))

import jax
import jax.numpy as jnp

## 1. The magic of JNP.EINSUM
`einsum` allows you to express complex tensor contractions using elegant index strings. Let's see how `simply` uses it to simplify attention projections.

In [27]:
# A standard batched matrix multiplication using matmul:
# Context: We have inputs [batch, seq, dim_in] and a weight matrix [dim_in, dim_out]
batch_size, seq_len, dim_in, dim_out = 2, 4, 8, 16
x = jnp.ones((batch_size, seq_len, dim_in))
w = jax.random.normal(jax.random.PRNGKey(0), (dim_in, dim_out))

# The traditional way (dot product handling axes manually)
out_traditional = jnp.dot(x, w)

# The EINSUM way (b=batch, s=seq, i=dim_in, o=dim_out)
# "bsi,io->bso"
# Contract the 'i' dimension. Keep 'b', 's', and 'o'.
out_einsum = jnp.einsum('bsi,io->bso', x, w)

print("Are they almost equal?", jnp.allclose(out_traditional, out_einsum))

Are they almost equal? True


## 2. Using simply's EinsumLinear
Because `einsum` is so powerful, `simply/utils/module.py` ships with `EinsumLinear`. This is the workhorse of their Transformer implementation!

Let's read `EinsumLinear` code or instantiate it directly.

In [28]:
from simply.utils.module import EinsumLinear

# Instead of passing 'in_features' and 'out_features', we pass an equation and shape!
# This allows incredibly flexible projection logic (e.g. mapping directly into heads).
linear = EinsumLinear(
    eqn='io,bsi->bso', 
    weight_shape=(8, 16), 
    bias_term='o' # Add bias to the output dimension 'o'
)

rng = jax.random.PRNGKey(1)
params = linear.init(rng)

output = linear.apply(params, x)
print("EinsumLinear inpurt shape:", x.shape)
print("EinsumLinear output shape:", output.shape)

EinsumLinear inpurt shape: (2, 4, 8)
EinsumLinear output shape: (2, 4, 16)


## Exercise: Projecting into Multiple Heads
In multi-head attention, we project an input `x` (shape `[batch, seq, model_dim]`) into queries `q` with shape `[batch, seq, num_heads, head_dim]`.

Write the `eqn` for an `EinsumLinear` to cleanly accomplish this projection! Use the characters `b` (batch), `s` (seq), `m` (model_dim), `h` (num_heads), `d` (head_dim).

In [29]:
# Write your solution here:
num_heads, head_dim, model_dim = 4, 64, 256
eqn = 'bsm,mhd->bshd'
weight_shape = (model_dim, num_heads, head_dim)

q_proj = EinsumLinear(
   eqn=eqn,
   weight_shape=weight_shape
)

### Solution:

In [30]:
num_heads, head_dim, model_dim = 4, 64, 256
x = jnp.ones((2, 10, model_dim))

# Equation:
# Input: b s m
# Weight: m h d
# Output: b s h d (Contract m)
q_proj = EinsumLinear(
    eqn='mhd,bsm->bshd',
    weight_shape=(model_dim, num_heads, head_dim)
)

params = q_proj.init(jax.random.PRNGKey(0))
q = q_proj.apply(params, x)
print("Query shape:", q.shape) # Should be (2, 10, 4, 64)

Query shape: (2, 10, 4, 64)
